## Gradiente de Brown Almost Linear (ALMOST)

La función **ALMOST** (Brown Almost Linear) se define como

$$f(x) = \sum_{i=1}^{n-1}\Big(x_i + S - (n+1)\Big)^2 + \Big(P - 1\Big)^2,$$

donde
$$S = \sum_{j=1}^{n} x_j, \qquad P = \prod_{j=1}^{n} x_j.$$

Definimos los residuos
$$r_i = x_i + S - (n+1)\quad (i=1,\dots,n-1), \qquad r_n = P - 1,$$
y denotamos $R = \sum_{i=1}^{n-1} r_i$.

Aplicando la regla de la cadena, para $k=1,\dots,n-1$ se cumple
$$\frac{\partial r_i}{\partial x_k} = \delta_{ik} + 1 \quad (i=1,\dots,n-1), \qquad \frac{\partial r_n}{\partial x_k} = P_k := \prod_{j\neq k} x_j,$$
y por tanto
$$\frac{\partial f}{\partial x_k} = 2\big(R + r_k\big) + 2\,r_n\,P_k \quad (k=1,\dots,n-1),$$
$$\frac{\partial f}{\partial x_n} = 2R + 2\,r_n\,P_n.$$

**Observación.** El término $P_k = P/x_k$ (producto de todas las variables excepto $x_k$) introduce un acoplamiento global entre todas las coordenadas: un cambio en una variable afecta a los $n-1$ residuos lineales y al término del producto. Esto genera valles curvos y estrechos donde el antigradiente no apunta directamente al mínimo.

In [1]:
import numpy as np

def almost_f(x):
    """
    Evalúa la función Brown Almost Linear (ALMOST).
    x: array de longitud n ≥ 1.
    """
    n = len(x)
    S = np.sum(x)
    r = x[:n-1] + S - (n+1)        # primeros n-1 residuos
    prod = np.prod(x)
    return np.sum(r**2) + (prod - 1.0)**2


def almost_grad(x):
    """
    Gradiente analítico de ALMOST.
    Devuelve un array de la misma dimensión que x.
    """
    n = len(x)
    S = np.sum(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)   # residuos lineales
    prod = np.prod(x)
    r[n-1] = prod - 1.0              # residuo del producto
    R = np.sum(r[:n-1])

    g = np.zeros(n)
    for k in range(n):
        # P_k = producto de todos los x_j con j != k (sin dividir por cero)
        if x[k] != 0:
            Pk = prod / x[k]
        else:
            Pk = np.prod(np.delete(x, k))

        if k < n-1:
            g[k] = 2.0 * (R + r[k]) + 2.0 * r[n-1] * Pk
        else:
            g[k] = 2.0 * R + 2.0 * r[n-1] * Pk
    return g

# ----- Verificación con n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])
print(f"f(x0)       = {almost_f(x0):.8f}")            # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")             # esperado: (-20.234375, -20.234375, -20.234375, -15.234375)
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.2153

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866


# Método Cuasi-Newton (BFGS) aplicado a ALMOST

## Gradiente de ALMOST (recordatorio)

$$f(x)=\sum_{i=1}^{n-1}\big(x_i+S-(n+1)\big)^2+(P-1)^2,\quad S=\sum_j x_j,\;P=\prod_j x_j.$$
Con $r_i=x_i+S-(n+1)$ ($i\le n-1$), $r_n=P-1$, $R=\sum_{i\le n-1}r_i$, $P_k=\prod_{j\neq k}x_j$:
$$\frac{\partial f}{\partial x_k}=2(R+r_k)+2r_n P_k\;(k<n),\qquad\frac{\partial f}{\partial x_n}=2R+2r_n P_n.$$

## Hessiana de ALMOST (recordatorio)

$$H_{kl}=2\sum_{i=1}^{n-1}(\delta_{ik}+1)(\delta_{il}+1)+2P_kP_l+2r_n\cdot\begin{cases}P_{kl},&k\neq l\\0,&k=l\end{cases}$$
En el óptimo \(x^*=(1,\dots,1)\), \(H\) es definida positiva (para \(n=4\): autovalores \(\approx 0.0871,\,2,\,2,\,45.913\)).

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

- $S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$.
- $f(x_0)\approx 19.6289$, $\nabla f(x_0)\approx(-20.2344,-20.2344,-20.2344,-15.2344)^t$, $\|\nabla f(x_0)\|\approx 38.2153$.
- $\nabla^2 f(x_0)$ es definida positiva (todos sus autovalores $>0$).
- $d_0=-B_0 g_0=-g_0=(20.2344,20.2344,20.2344,15.2344)^t$.
- La búsqueda lineal de Wolfe acepta $\alpha_0=1$ en la primera iteración (la dirección apunta hacia $(1,1,1,1)$).
- Se calculan $s_0$, $y_0$ y se actualiza $B_1$ mediante BFGS.

## Condiciones de aplicabilidad

1. $f\in C^1(\mathbb{R}^n)$: se cumple ($f$ es polinomio, $C^\infty$).
2. $B_0$ simétrica y definida positiva: $B_0=I$ cumple.
3. Búsqueda lineal que garantice $y_k^t s_k>0$: se satisface con Wolfe ($0<c_1<c_2<1$).
4. Dirección \(d_k=-B_k g_k\) de descenso: $g_k^t d_k=-g_k^t B_k g_k<0$ porque $B_k$ es definida positiva.

In [2]:
def almost_f(x):
    """Función ALMOST."""
    n = len(x); S = np.sum(x)
    r = x[:n-1] + S - (n+1)
    return np.sum(r**2) + (np.prod(x) - 1.0)**2

def almost_grad(x):
    """Gradiente analítico de ALMOST."""
    n = len(x); S = np.sum(x); P = np.prod(x)
    r = np.zeros(n)
    r[:n-1] = x[:n-1] + S - (n+1)
    r[n-1] = P - 1.0
    R = np.sum(r[:n-1])
    Pk = np.array([P/x[k] if x[k] != 0 else np.prod(np.delete(x, k)) for k in range(n)])
    g = np.zeros(n)
    for k in range(n-1):
        g[k] = 2*(R + r[k]) + 2*r[n-1]*Pk[k]
    g[n-1] = 2*R + 2*r[n-1]*Pk[n-1]
    return g

# bfgs() ya definida en la sección SINGX; se reutiliza aquí.

In [3]:
# ----- Ejecución: BFGS sobre ALMOST, n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])

print(f"f(x0)       = {almost_f(x0):.8f}")       # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")        # esperado: [-20.234375 -20.234375 -20.234375 -15.234375]
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.215300
print()

x_opt, hist = bfgs(almost_f, almost_grad, x0, tol=1e-12, max_iter=200,
                   verbose=True, print_every=2)

print(f"\n--- Resultado final ---")
print(f"x_opt       = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)    = {almost_f(x_opt):.3e}")  # esperado: → 0 (< 1e-20)
print(f"||grad||    = {np.linalg.norm(almost_grad(x_opt)):.3e}")
print(f"Iteraciones = {len(hist['f'])-1}")     # esperado: 5-15
print(f"f tras 1ª iteración = {hist['f'][1]:.4e}")  # esperado: ≈ 10^{-2} -- 10^{-4}

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866



NameError: name 'bfgs' is not defined